# Working with Redis for Caching

This notebook demonstrates how to use the `RedisCache` and `RedisSemanticCache` classes from the langchain-redis package to implement caching for LLM responses using Redis.

## Installation

In [ ]:
# %pip install ipywidgets
# %pip install langchain-core
# %pip install langchain-redis
# %pip install langchain-openai
# %pip install redis

## Importing Required Libraries

In [1]:
import os
import time
import redis

from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_redis import RedisCache, RedisSemanticCache

## Setting up Redis Connection

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

redis_url = os.getenv("REDIS_URL")
redis_client = redis.from_url(redis_url)
redis_client.ping()

True

## Set the OpenAI API key

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

## Using Redis as a Standard Cache

In [ ]:
redis_cache = RedisCache(redis_url=redis_url)
set_llm_cache(redis_cache)

llm = OpenAI(temperature=0)

def execute_with_timing(prompt):
    start_time = time.time()
    result = llm.invoke(prompt)
    end_time = time.time()
    return result, end_time - start_time

# First call (not cached)
prompt = "Explain the concept of caching in three sentences."
result1, time1 = execute_with_timing(prompt)
print(f"First call (not cached):")
print(f"{result1}\nTime: {time1:.2f} seconds\n")

# Second call (should be cached)
result2, time2 = execute_with_timing(prompt)
print(f"Second call (cached):")
print(f"{result2}\nTime: {time2:.2f} seconds\n")

print(f"Speed improvement: {time1 / time2:.2f}x faster")

# Clear the cache
redis_cache.clear()
print("Cache cleared")

First call (not cached):


Caching is the process of storing frequently accessed data in a temporary storage location for faster retrieval. This helps to reduce the time and resources needed to access the data from its original source. Caching is commonly used in computer systems, web browsers, and databases to improve performance and efficiency.
Time: 0.88 seconds

Second call (cached):


Caching is the process of storing frequently accessed data in a temporary storage location for faster retrieval. This helps to reduce the time and resources needed to access the data from its original source. Caching is commonly used in computer systems, web browsers, and databases to improve performance and efficiency.
Time: 0.02 seconds

Speed improvement: 50.15x faster
Cache cleared


c:\Users\samin\Python\REDIS\langchain-apps-with-redis-main\langchain-apps-with-redis-main\.venv\Lib\site-packages\langchain_redis\cache.py:227: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(json.dumps(result))]


## Using Redis as a Semantic Cache

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
semantic_cache = RedisSemanticCache(
    redis_url=redis_url, embeddings=embeddings, distance_threshold=0.2
)

set_llm_cache(semantic_cache)

# Original prompt
original_prompt = "What is the capital of France?"
result1, time1 = execute_with_timing(original_prompt)
print(f"Original query:\nPrompt: {original_prompt}\n")
print(f"{result1}\nTime: {time1:.2f} seconds\n")

# Semantically similar prompt
similar_prompt = "Can you tell me the capital city of France?"
result2, time2 = execute_with_timing(similar_prompt)
print(f"Similar query:\nPrompt: {similar_prompt}\n")
print(f"{result2}\nTime: {time2:.2f} seconds\n")

print(f"Speed improvement: {time1 / time2:.2f}x faster")

Original query:
Prompt: What is the capital of France?



The capital of France is Paris.
Time: 1.84 seconds

Similar query:
Prompt: Can you tell me the capital city of France?



The capital of France is Paris.
Time: 0.31 seconds

Speed improvement: 6.00x faster


c:\Users\samin\Python\REDIS\langchain-apps-with-redis-main\langchain-apps-with-redis-main\.venv\Lib\site-packages\langchain_redis\cache.py:496: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  loads(gen_str)


## Cleanup

In [12]:
# Clear the semantic cache
semantic_cache.clear()
print("Semantic cache cleared")

Semantic cache cleared
